# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [4]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [5]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [10]:
#
print(df.shape)
print('\n')
print(df.dtypes)
print('\n')
print(df.isna().sum())
print('\n')
print(df.duplicated().sum())

(8, 6)


order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object


order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64


1


**What is wrong with this data?** List at least five specific problems:

Sorry im like lobotomized rn

1. Order 1 is duplicated
2. price stored as object
3. qty null order 3
4. item inconsistent typography
5. three date formats

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [24]:
 removed = int(df.duplicated().sum())
clean = df.drop_duplicates().copy()
log('duplicates', 'dropped exact duplicate rows', removed)

# TODO: log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [18]:
clean['price'] = (clean['price'].astype(str)
                                .str.replace('$', '', regex=False)
                                .str.strip()
                                .astype(float))

assert clean['price'].dtype == float
log('price', 'price arrived as text with $; stripped and cast to float', len(clean))

[price] price arrived as text with $; stripped and cast to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [25]:
# TODO: clean['qty'] = pd[.to_numeric(...)
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = int(clean['qty'].isna().sum())    # TODO: count of NaN quantities
negative = int((clean['qty'] < 0).sum())   # TODO: count of negative quantities

# TODO: apply your decision, then log both separately
clean = clean[clean['qty'].notna()].copy()
log('qty missing', 'dropped no quanitity rows','missing')

clean = clean[clean['qty'] >= 0].copy()
log('qty-refund', 'dropped refund rows (negative qty); reporting gross sales', negative)



[qty missing] dropped no quanitity rows (missing row(s))
[qty-refund] dropped refund rows (negative qty); reporting gross sales (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [26]:
print('before:', sorted(clean['category'].unique()))

# TODO: lowercase, strip, remove punctuation
# TODO: CATEGORY_MAP = {...} for the judgment calls
print('before:', sorted(clean['category'].unique()))
n_before = clean['category'].nunique()

clean['category'] = (clean['category'].str.lower()
                                      .str.strip()
                                      .str.replace(r'[^a-z0-9]', '', regex=True))

CATEGORY_MAP = {'raingear': 'rain gear'}   # judgment call: readable label
clean['category'] = clean['category'].replace(CATEGORY_MAP)

# print('after: ', sorted(clean['category'].unique()))


print('after: ', sorted(clean['category'].unique()))
log('category', f'normalized case/punctuation, mapped variants: {n_before} -> {clean["category"].nunique()} categories', len(clean))

before: ['Apparel', 'Food', 'Merch', 'food', 'rain-gear']
before: ['Apparel', 'Food', 'Merch', 'food', 'rain-gear']
after:  ['apparel', 'food', 'merch', 'rain gear']
[category] normalized case/punctuation, mapped variants: 5 -> 4 categories (5 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [27]:

print('before:', clean['item'].tolist())

clean['item'] = (clean['item'].str.lower()
                              .str.strip()
                              .str.replace(r'\s+', ' ', regex=True))

ITEM_MAP = {'cheese burger': 'cheeseburger'}
clean['item'] = clean['item'].replace(ITEM_MAP)

no_item = int(clean['item'].isna().sum())
clean['item'] = clean['item'].fillna('unknown')   # keep the row: qty/price are valid, revenue is real

print('after: ', clean['item'].tolist())
log('item', 'normalized names, merged spelling variants, filled missing item with "unknown"', no_item)

before: ['Cheeseburger', 'cheese burger', 'UVA T-Shirt ', 'rain poncho', nan]
after:  ['cheeseburger', 'cheeseburger', 'uva t-shirt', 'rain poncho', 'unknown']
[item] normalized names, merged spelling variants, filled missing item with "unknown" (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [28]:
# TODO
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce', format='mixed')

failed = int(clean['ts'].isna().sum())
print('unparseable timestamps:', failed)

clean['hour'] = clean['ts'].dt.hour
log('timestamps', 'parsed ts to datetime (mixed formats), failures -> NaT; added hour', failed)

unparseable timestamps: 1
[timestamps] parsed ts to datetime (mixed formats), failures -> NaT; added hour (1 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [32]:
clean['price'] = (clean['price'].astype(str)
                                .str.replace('$', '', regex=False)
                                .str.strip()
                                .astype(float))

# TODO: assertions
assert clean.duplicated().sum() == 0
assert clean['price'].dtype == float
assert (clean['price'] > 0).all()
assert clean['qty'].notna().all() and (clean['qty'] >= 0).all()
assert clean['category'].str.islower().all()
assert clean['item'].notna().all()
assert pd.api.types.is_datetime64_any_dtype(clean['ts'])

# TODO: clean['revenue'] =
clean['revenue'] = clean['qty'] * clean['price']

# TODO: print rows, units, revenue, distinct categories
print('rows:               ', len(clean))
print('units:              ', clean['qty'].sum())
print('revenue:            ', clean['revenue'].sum())
print('distinct categories:', clean['category'].nunique())

rows:                5
units:               10.0
revenue:             106.5
distinct categories: 4


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [30]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,price arrived as text with $; stripped and cas...,7
2,qty missing,dropped no quanitity rows,missing
3,duplicates,dropped exact duplicate rows,1
4,qty missing,dropped no quanitity rows,missing
5,qty-refund,dropped refund rows (negative qty); reporting ...,1
6,category,"normalized case/punctuation, mapped variants: ...",5
7,item,"normalized names, merged spelling variants, fi...",1
8,timestamps,"parsed ts to datetime (mixed formats), failure...",1


**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [33]:
rows_after = len(clean)                                   # 5
revenue_after = clean['revenue'].sum()                    # 106.5
biggest_decision = 'excluding the refund (negative qty) from revenue'
revenue_other_way = revenue_after + (-3 * 6.0)            # 88.5

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 106.5
decision that mattered: excluding the refund (negative qty) from revenue
revenue the other way: 88.5
